# 81. 子图、控件与导出

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 18 / 18 步：组合控件、子图并完成交付**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 地图图表（Map / Geo）  →  **本章任务：** 子图、控件与导出  →  **下一步：** 模块大作业《周度经营预警会：交互诊断与行动看板》
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

做数据分析时经常不只画一张图，而是想在一个画面里同时对比趋势和排名等多个角度。



## 本章目标

学完本章，你将能够：

- **理解**：理解「子图、控件与导出」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「子图、控件与导出」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「子图、控件与导出」并读出其中的结论。


## 81.1 适用场景

**背景引入**：做数据分析时经常不只画一张图，而是想在一个画面里同时对比趋势和排名等多个角度。Plotly 能把这些相关图表放进同一个 Figure，让读者缩放、悬停、切换时互不干扰，一眼看清多组数据之间的关系；学完这些操作，你的报告就不再是一张张孤立的图片，而是能直接回答“它们之间到底有什么联系”。（把多张图“嵌进同一个画框”就是子图：先用 make_subplots 画好格子，再把每张图放进对应格子，读者在同一画布上缩放、切换、互不干扰——就像把几张桌面拼成一张老板桌。）

多个相关图表需要在一个Figure中协调展示或切换。


## 81.2 数据结构

共享维度的多组数据；导出前应控制Trace数量。


## 81.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 updatemenus 的 direction 从 "down" 改为 "right"，观察按钮排列方向的变化
2. 修改 rangeslider_visible 从 True 为 False，对比有无范围滑块的交互差异
3. 将 to_html 的 include_plotlyjs 从 "cdn" 改为 True，说明内联脚本对HTML文件大小的影响


## 81.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `fig.add_trace()`、`go.Scatter()`、`regional.groupby()`、`go.Bar()` | 多个相关图表需要在一个Figure中协调展示或切换。 | 控件太多 |
| 进阶变体 | `go.Figure()`、`fig.add_trace()`、`go.Scatter()`、`fig.update_layout()` | 在基础图表上增加分组、注释、布局或交互 | 按钮状态与标题不同步 |
| 关键参数 | `make_subplots` | 布局 | 控件太多 |
| 关键参数 | `updatemenus` | 按钮 | 按钮状态与标题不同步 |
| 关键参数 | `rangeslider` | 范围滑块 | HTML过大 |
| 关键参数 | `to_html` | 导出 | 子图图例重复 |


## 81.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(f"Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行")


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 81.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["销售趋势", "区域销售"])
fig.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["sales"],
        mode="lines+markers",
        name="销售额",
    ),
    row=1,
    col=1,
)
totals = regional.groupby("region", as_index=False)["sales"].sum()
fig.add_trace(
    go.Bar(x=totals["region"], y=totals["sales"], name="区域合计"), row=1, col=2
)
fig.update_layout(title="经营分析组合图", template="plotly_white")
fig.show()


**练一练**：对 74.4 的基础图表做两处小改动，观察图形变化。
1. 把组合图标题从 `"经营分析组合图"` 改成你喜欢的名字（比如 `"经营分析组合图（练习版）"`）；
2. 把左子图“销售额”曲线的连线方式从 `"lines+markers"`（点线都画）改为只显示数据点（`mode="markers"`）。
运行后对比：右子图的柱状图有没有受影响？标题改了之后，坐标轴标题和单位还和图表对得上吗？在下方代码的 `new_title` 与 `new_mode` 处填上你的取值再运行。


In [ ]:
# 请在下方填写代码
# 任务：修改 74.4 基础图表 ——
#   1) 把组合图标题改成新标题（填到 new_title）
#   2) 把左子图“销售额”曲线的连线方式改为只显示数据点 mode="markers"（填到 new_mode）
# 参考：图对象 fig 已在上方单元格创建，可用 fig.update_layout(title=...) 与
# fig.update_traces(...)。

new_title = ___TITLE___  # 例如："经营分析组合图（练习版）"
new_mode = ___MODE___  # 例如："markers"

fig.update_layout(title=new_title)
fig.update_traces(mode=new_mode)
fig.show()


In [ ]:
new_title = "经营分析组合图（练习版）"
new_mode = "markers"

# 修改标题
fig.update_layout(title=new_title)
# 只调整第一条 trace（左子图“销售额”散点）的连线方式，右子图柱状图保留不动
fig.data[0].update(mode=new_mode)
fig.show()


## 81.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["sales"],
        mode="lines+markers",
        name="销售额",
        visible=True,
    )
)
fig.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["profit"],
        mode="lines+markers",
        name="利润",
        visible=False,
    )
)
fig.update_layout(
    title="指标切换",
    updatemenus=[
        {
            "buttons": [
                {
                    "label": "销售额",
                    "method": "update",
                    "args": [{"visible": [True, False]}, {"title": "月度销售额"}],
                },
                {
                    "label": "利润",
                    "method": "update",
                    "args": [{"visible": [False, True]}, {"title": "月度利润"}],
                },
            ],
            "direction": "down",
        }
    ],
    template="plotly_white",
)
fig.show()


## 81.8 参数说明

- make_subplots：布局
- updatemenus：按钮
- rangeslider：范围滑块
- to_html：导出


## 81.9 结果解读

控件应解决明确任务，默认状态必须可读，交互变化需要保持单位和标题一致。


## 81.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 81.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 81.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 81.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 81.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 81.12 易错点提醒

- 控件太多
- 按钮状态与标题不同步
- HTML过大
- 子图图例重复


## 81.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 81.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：增加一个控件，让图表可交互切换
# 【目标】给子图加 range slider，让时间范围可拖动，练习交互。
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 起点示例(已可运行)：给折线加 range slider。
fig = make_subplots(rows=1, cols=2, subplot_titles=["销售趋势", "区域销售"])
fig.add_trace(
    go.Scatter(x=monthly["month"], y=monthly["sales"], mode="lines+markers", name="销售额"),
    row=1,
    col=1,
)
totals = regional.groupby("region", as_index=False)["sales"].sum()
fig.add_trace(go.Bar(x=totals["region"], y=totals["sales"], name="区域销售"), row=1, col=2)
fig.update_layout(title="经营看板（含滑块）", xaxis=dict(rangeslider=dict(visible=True)))
fig.show()

# ---- 反思记录：加上滑块后，交互带来了什么 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
fig = px.line(monthly, x="month", y="sales", markers=True, title="可导出的销售趋势")
fig.update_xaxes(rangeslider_visible=True)
html = fig.to_html(include_plotlyjs="cdn", full_html=True)
print(f"HTML字符数: {len(html):,}")
fig.show()


## 81.15 小结

组合子图、按钮、下拉菜单、范围控件和HTML导出，形成完整交互视图。


### 81.15.1 你已经掌握

- 判断子图、控件与导出的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 81.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `make_subplots` | 布局 |
| `updatemenus` | 按钮 |
| `rangeslider` | 范围滑块 |
| `to_html` | 导出 |


### 81.15.3 需要注意

- 控件太多
- 按钮状态与标题不同步
- HTML过大
- 子图图例重复


### 81.15.4 完成检查

- [ ] 能判断什么问题适合使用子图、控件与导出
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 81.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
